# Gulati Conformer, small LibriSpeech CTC run

Trains the Conformer encoder from Gulati et al., Interspeech 2020, with a character CTC head. The Colab runtime should be a **T4 GPU** (`Runtime` → `Change runtime type` → `T4 GPU`).

The paper decodes with an RNN-Transducer. CTC is the piece that fits a free 16 GB T4 and still produces **WER** and **CER**.

One block, implemented in `src/conformer_asr/encoder.py`:

```text
x = x + 1/2 FFN(x)
x = x + MHSA(x)    relative sinusoidal positions
x = x + Conv(x)    GLU, depthwise conv, batch norm, Swish
x = x + 1/2 FFN(x)
x = LayerNorm(x)
```

Data is a duration-filtered slice of `openslr/librispeech_asr` (clean, CC-BY-4.0): 1000 train and 200 validation utterances, not the full 100 hours. Expect the untrained CER to sit near 1.0. A working run moves validation CER down and prints both scores at the end.

Use a fresh Colab runtime (`Runtime` → `Disconnect and delete runtime`) so this notebook clones the current repo instead of an old `/content/conformer-scratch` checkout. The GitHub repo must be public.

## TOKEN HERE

Replace every placeholder in the next cell. Do not commit the filled-in notebook.

- WandB key: https://wandb.ai/authorize
- Hugging Face token with **write** access: https://huggingface.co/settings/tokens
- `HF_REPO_ID`: `<huggingface-username>/conformer-asr-tiny` (created for you, private)
- The GitHub URL is already set to `https://github.com/rupaktiwari/conformer-scratch.git`

In [ ]:
# ============================================================
# TOKEN HERE
# ============================================================
WANDB_API_KEY = "TOKEN HERE"
HF_TOKEN = "TOKEN HERE"

GITHUB_REPO = "https://github.com/rupaktiwari/conformer-scratch.git"
HF_REPO_ID = "YOUR_HF_USERNAME/conformer-asr-tiny"
WANDB_PROJECT = "conformer-asr"
WANDB_ENTITY = ""  # leave empty to use your personal WandB account
# ============================================================

import os

def _reject(name, value, banned):
    missing = (not value) or any(piece in value for piece in banned)
    if missing:
        raise SystemExit(f"Fill in {name} in the TOKEN HERE cell.")

_reject("WANDB_API_KEY", WANDB_API_KEY, ["TOKEN HERE"])
_reject("HF_TOKEN", HF_TOKEN, ["TOKEN HERE"])
_reject("GITHUB_REPO", GITHUB_REPO, ["YOUR_GITHUB_USERNAME", "TOKEN HERE"])
_reject("HF_REPO_ID", HF_REPO_ID, ["YOUR_HF_USERNAME"])

os.environ["WANDB_API_KEY"] = WANDB_API_KEY
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HF_REPO_ID"] = HF_REPO_ID
os.environ["WANDB_PROJECT"] = WANDB_PROJECT
if WANDB_ENTITY.strip():
    os.environ["WANDB_ENTITY"] = WANDB_ENTITY.strip()

print("Tokens are set in this runtime. They are not written to disk by this cell.")

In [ ]:
import os
import shutil
import subprocess
import sys

import torch

if not torch.cuda.is_available():
    raise SystemExit("No GPU visible. Runtime -> Change runtime type -> T4 GPU, then rerun.")
print("GPU:", torch.cuda.get_device_name(0))

repo_dir = "/content/conformer-scratch"
if os.path.isdir(repo_dir):
    shutil.rmtree(repo_dir)
subprocess.check_call(["git", "clone", "--depth", "1", GITHUB_REPO, repo_dir])

# only-if-needed keeps the CUDA torch that Colab already installed.
subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed", "-e", repo_dir]
)
os.chdir(repo_dir)
# Editable installs are registered in a .pth file that Python reads only at
# startup. Put src on the path so this kernel can import conformer_asr now.
src_dir = os.path.join(repo_dir, "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)
import conformer_asr
print("Installed conformer-asr from", repo_dir)
print("Importing", conformer_asr.__file__)

## Shapes before the long run

One forward pass of the encoder. Log-mel input is `(batch, time, 80)`. Two stride-2 convolutions shorten time by about 4x.

In [ ]:
import os
import sys
import torch
from dataclasses import asdict

src_dir = "/content/conformer-scratch/src"
if os.path.isdir(src_dir) and src_dir not in sys.path:
    sys.path.insert(0, src_dir)

from conformer_asr import ConformerEncoder, count_parameters
from conformer_asr.config import load_config

cfg = load_config("configs/small.yaml")
encoder = ConformerEncoder(**asdict(cfg.model)).cuda().eval()
feats = torch.randn(2, 200, cfg.model.n_mels, device="cuda")
lengths = torch.tensor([200, 120], device="cuda")
hidden, hidden_lengths = encoder(feats, lengths)
print("input ", tuple(feats.shape))
print("hidden", tuple(hidden.shape), "lengths", hidden_lengths.tolist())
print("parameters", f"{count_parameters(encoder):,}")
del encoder, feats, hidden

## Train

The next cell streams the LibriSpeech subset, caches log-mel features under `data/cache/`, then trains. The first run spends several minutes on download and feature extraction. Later runs in the same runtime reuse the cache.

Logged to WandB (`conformer-asr`) and to the Hugging Face repo in `HF_REPO_ID` (`metrics.jsonl` each validation, `conformer_ctc.pt` at the end).

In [ ]:
from conformer_asr.train import run

result = run("configs/small.yaml")

print()
print(f"Untrained  WER {result['wer_untrained']:.3f}   CER {result['cer_untrained']:.3f}")
print(f"Best ckpt  WER {result['wer']:.3f}   CER {result['cer']:.3f}")
print(
    f"Train probe WER {result['train_sample_wer']:.3f}   CER {result['train_sample_cer']:.3f}"
)
print("checkpoint:", result["checkpoint"])
print("hf repo:   ", result["hf_repo_id"])
print()
for item in result["examples"]:
    print("REF:", item["ref"])
    print("HYP:", item["hyp"])
    print("---")